In [11]:
%%file producer.py
from kafka import KafkaProducer
import json, random, time
from datetime import datetime

producer = KafkaProducer(
    bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

sklepy = ['Warszawa', 'Kraków', 'Gdańsk', 'Wrocław']
kategorie = ['elektronika', 'odzież', 'żywność', 'książki']

def generate_transaction():
    return {
        'tx_id': f'TX{random.randint(1000,9999)}',
        'user_id': f'u{random.randint(1,20):02d}',
        'amount': round(random.uniform(5.0, 5000.0), 2),
        'store': random.choice(sklepy),
        'category': random.choice(kategorie),
        'timestamp': datetime.now().isoformat(),
    }

for i in range(100):
    tx = generate_transaction()
    producer.send('transactions', value=tx)
    print(f"[{i+1}] {tx['tx_id']} | {tx['amount']:.2f} PLN | {tx['store']}")
    time.sleep(0.5)

producer.flush()
producer.close()

Writing producer.py


In [12]:
%%file producer.py
from kafka import KafkaProducer
import json, random, time
from datetime import datetime

producer = KafkaProducer(
    bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

sklepy = ['Warszawa', 'Kraków', 'Gdańsk', 'Wrocław']
kategorie = ['elektronika', 'odzież', 'żywność', 'książki']

def generate_transaction():
    if random.random() < 0.05:  # 5% podejrzanych transakcji
        return {
            'tx_id': f'TX{random.randint(1000,9999)}',
            'user_id': f'u{random.randint(1,20):02d}',
            'amount': round(random.uniform(3000.01, 5000.0), 2),
            'store': random.choice(sklepy),
            'category': 'elektronika',
            'hour': random.randint(0, 5),
            'timestamp': datetime.now().isoformat(),
        }
    else:
        return {
            'tx_id': f'TX{random.randint(1000,9999)}',
            'user_id': f'u{random.randint(1,20):02d}',
            'amount': round(random.uniform(5.0, 5000.0), 2),
            'store': random.choice(sklepy),
            'category': random.choice(kategorie),
            'hour': random.randint(6, 23),  # zwykłe transakcje poza nocą
            'timestamp': datetime.now().isoformat(),
        }

for i in range(1000):
    tx = generate_transaction()
    producer.send('transactions', value=tx)
    print(f"[{i+1}] {tx['tx_id']} | {tx['amount']:.2f} PLN | {tx['store']} | {tx['category']} | hour={tx['hour']}")
    time.sleep(0.5)

producer.flush()
producer.close()

Overwriting producer.py


In [13]:
%%file consumer_filter.py
from kafka import KafkaConsumer
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

for message in consumer:
    tx = message.value
    
    if tx['amount'] > 1000:
        print(f"ALERT: {tx['tx_id']} | {tx['amount']:.2f} PLN | {tx['store']} | {tx['category']}")

Writing consumer_filter.py


In [14]:
%%file consumer_enrich.py
from kafka import KafkaConsumer
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    value_deserializer=lambda x: json.loads(x.decode('utf-8')),
    auto_offset_reset='earliest',
    enable_auto_commit=True,
    group_id='consumer-enrich-group'
)

for message in consumer:
    tx = message.value
    amount = tx['amount']

    if amount > 3000:
        risk_level = "HIGH"
    elif amount > 1000:
        risk_level = "MEDIUM"
    else:
        risk_level = "LOW"

    print(
        f"TX: {tx['tx_id']} | "
        f"Kwota: {amount:.2f} PLN | "
        f"Sklep: {tx['store']} | "
        f"Kategoria: {tx['category']} | "
        f"Risk: {risk_level}"
    )

Writing consumer_enrich.py


In [16]:
%%file consumer_count.py
from kafka import KafkaConsumer
from collections import Counter, defaultdict
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    auto_offset_reset='earliest',
    group_id='count-group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

store_counts = Counter()
total_amount = defaultdict(float)
msg_count = 0

for message in consumer:
    tx = message.value
    store = tx['store']
    amount = tx['amount']

    store_counts[store] += 1
    total_amount[store] += amount
    msg_count += 1

    if msg_count % 10 == 0:
        print(f"\n--- PODSUMOWANIE PO {msg_count} WIADOMOŚCIACH ---")
        print(f"{'Sklep':<12} {'Liczba transakcji':<20} {'Suma kwot [PLN]':<20}")

        for store_name in store_counts:
            print(f"{store_name:<12} {store_counts[store_name]:<20} {total_amount[store_name]:<20.2f}")

Overwriting consumer_count.py


In [17]:
def score_transaction(tx):
    score = 0
    rules = []

    # R1: amount > 3000 -> +3
    if tx['amount'] > 3000:
        score += 3
        rules.append('R1')

    # R2: elektronika i amount > 1500 -> +2
    if tx['category'] == 'elektronika' and tx['amount'] > 1500:
        score += 2
        rules.append('R2')

    # R3: godzina < 6 -> +2
    hour = int(tx['timestamp'][11:13])
    if hour < 6:
        score += 2
        rules.append('R3')

    return score, rules


# Test
test_tx = {
    'tx_id': 'TX999',
    'amount': 4500.0,
    'category': 'elektronika',
    'timestamp': '2026-04-01T03:15:00'
}

print(score_transaction(test_tx))

(7, ['R1', 'R2', 'R3'])


In [18]:
%%file scoring_consumer.py
from kafka import KafkaConsumer, KafkaProducer
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    auto_offset_reset='earliest',
    group_id='scoring-group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

alert_producer = KafkaProducer(
    bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

def score_transaction(tx):
    score = 0
    rules = []

    # R1: amount > 3000 -> +3
    if tx['amount'] > 3000:
        score += 3
        rules.append('R1')

    # R2: elektronika i amount > 1500 -> +2
    if tx['category'] == 'elektronika' and tx['amount'] > 1500:
        score += 2
        rules.append('R2')

    # R3: godzina < 6 -> +2
    if 'hour' in tx:
        hour = tx['hour']
    else:
        hour = int(tx['timestamp'][11:13])

    if hour < 6:
        score += 2
        rules.append('R3')

    return score, rules

for message in consumer:
    tx = message.value
    score, rules = score_transaction(tx)

    if score >= 3:
        alert = {
            'tx_id': tx['tx_id'],
            'amount': tx['amount'],
            'store': tx['store'],
            'category': tx['category'],
            'timestamp': tx['timestamp'],
            'score': score,
            'rules': rules,
            'status': 'PODEJRZANA'
        }

        if 'hour' in tx:
            alert['hour'] = tx['hour']

        alert_producer.send('alerts', value=alert)
        alert_producer.flush()

        print(
            f"ALERT: {alert['tx_id']} | "
            f"{alert['amount']:.2f} PLN | "
            f"{alert['store']} | "
            f"{alert['category']} | "
            f"score={alert['score']} | "
            f"rules={alert['rules']}"
        )

Writing scoring_consumer.py


In [19]:
%%file alerts_consumer.py
from kafka import KafkaConsumer
import json

consumer = KafkaConsumer(
    'alerts',
    bootstrap_servers='broker:9092',
    auto_offset_reset='earliest',
    group_id='alerts-group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

for message in consumer:
    alert = message.value
    print(alert)

Writing alerts_consumer.py
